In [1]:
import json

def json_to_plain_text(json_file_path, output_txt_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    def extract_text(obj):
        if isinstance(obj, dict):
            # Recursively extract from dict values
            return '\n'.join(extract_text(v) for v in obj.values())
        elif isinstance(obj, list):
            # Recursively extract from list items
            return '\n'.join(extract_text(i) for i in obj)
        elif isinstance(obj, str):
            return obj
        else:
            return ''

    plain_text = extract_text(data)

    with open(output_txt_path, 'w', encoding='utf-8') as f:
        f.write(plain_text)

# Usage example:
json_to_plain_text('filtered.json', 'output.txt')


In [2]:
with open('data.txt', 'r', encoding='utf-8') as f1, \
     open('output.txt', 'r', encoding='utf-8') as f2, \
     open('combined.txt', 'w', encoding='utf-8') as out:
    out.write(f1.read())
    out.write('\n')  # Optional: add newline between files
    out.write(f2.read())


In [8]:

token_counts = {}
with open("combined.txt", "r", encoding="utf-8") as f:
    for line in f:
        tokens = tokenizer.encode(line).tokens
        for t in tokens:
            token_counts[t] = token_counts.get(t, 0) + 1

# Sort tokens by frequency
sorted_tokens = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)

# Calculate cumulative coverage for different vocab sizes
total_tokens = sum(token_counts.values())
cumulative = 0
for i, (token, freq) in enumerate(sorted_tokens):
    cumulative += freq
    coverage = cumulative / total_tokens
    if i+1 in [10000, 20000, 32000, 50257]:
        print(f"Vocab size: {i+1}, coverage: {coverage:.4f}")


Vocab size: 10000, coverage: 0.9495
Vocab size: 20000, coverage: 0.9902


In [5]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Step 1: Initialize an empty BPE tokenizer
tokenizer = Tokenizer(models.BPE())

# Step 2: Use a simple whitespace-based pre-tokenizer (adjust as needed)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Step 3: Define a trainer for BPE
trainer = trainers.BpeTrainer(
    vocab_size=32000,         # Set desired vocabulary size
    min_frequency=5,          # Minimum frequency for a token to be included
    special_tokens=["<|endoftext|>","<|user|>","<|assistant|>"]
)

# Step 4: Train on your dataset
tokenizer.train(["output.txt"], trainer)

# Step 5: Save the trained tokenizer
tokenizer.save("guj_bpe_tokenizer.json")

In [1]:
import torch

device = 'cpu'
if torch.cuda.is_available():
    device="cuda"
    
print(f"Using device {device}")

Using device cpu


In [2]:
from tokenizers import Tokenizer
from gpt2 import *
# Load the trained custom tokenizer
tokenizer = Tokenizer.from_file("guj_bpe_tokenizer.json")


In [ ]:
BASE_CONFIG = {
    "vocab_size": 32000,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.1,        # Dropout rate
    "qkv_bias": True,
    "emb_dim": 768, 
    "n_layers": 12, 
    "n_heads": 12      
}

with torch.device("meta"):
    model = GPTModel(BASE_CONFIG)

model.load_state_dict(
    torch.load("best_model.pth", map_location=device, weights_only=True, mmap=True),
    assign=True
)

In [7]:
import random 

with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()  # one big string

words = text.splitlines()  # split into list of words

# print(words[:10])
random.seed(42)
random.shuffle(words)
# print(words[:10])
data = '\n'.join(words)

In [8]:

# Let's assume `data` is a list of strings or tokenized examples
split_ratio = 0.9  # 90% train, 10% test
split_index = int(len(data) * split_ratio)

train_data = data[:split_index]
test_data = data[split_index:]

train_loader = create_dataloader_v1(train_data, 16, 1024, 512)
test_loader = create_dataloader_v1(test_data, 16, 1024, 512,shuffle=False,drop_last=False)


In [ ]:
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim import AdamW
import os

# Model and optimizer setup
optimizer = AdamW(model.parameters(), lr=1e-5)
model.to(device)
epochs = 10
# Hyperparameters
warmup_steps = 1000  # Adjust this based on total steps
max_grad_norm = 1.0  # Gradient clipping
total_steps = len(train_loader) * epochs  # Total training steps
patience = 3  # How many epochs to wait for improvement
best_loss = float('inf')  # Start with a very high best loss
counter = 0  # Counter to track epochs without improvement

# Cosine decay scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=0)

# Directory to save the model
save_dir = 'saved_models'
os.makedirs(save_dir, exist_ok=True)

# Training loop
for epoch in range(epochs):
    model.train()
    train_loss = 0

    # Learning rate warmup
    warmup_rate = min(1.0, (epoch * len(train_loader) + 1) / warmup_steps)  # Linearly scale
    for step, (x, y) in enumerate(train_loader):
        optimizer.zero_grad()

        # Autocast for mixed precision training
        with torch.autocast(device, dtype=torch.bfloat16):
            loss = calc_loss_batch(x, y, model, device)

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        # Optimizer step
        optimizer.step()

        train_loss += loss.item()

    # Scheduler step (after warmup phase)
    scheduler.step()

    avg_train_loss = train_loss / len(train_loader)

    # Evaluation phase
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for x, y in test_loader:
            with torch.autocast(device, dtype=torch.bfloat16):
                loss = calc_loss_batch(x, y, model, device)
            test_loss += loss.item()

    avg_test_loss = test_loss / len(test_loader)

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Test Loss = {avg_test_loss:.4f}")

    # Early Stopping: If test loss hasn't improved for 'patience' epochs
    if avg_test_loss < best_loss:
        best_loss = avg_test_loss
        counter = 0  # Reset counter as the model improved
        # Save the model
        torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))
        print("Model saved.")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs without improvement.")
            break  # Stop training



In [ ]:
generate_and_print_sample(model,tokenizer,device,start_context="જે એપેક્સ બેંક પર 50 કરોડના કૌભાંડના આરોપો")